# ⚡ Módulo 10 — Orchestration: Concurrent

> **Objetivo:** Ejecutar varios agentes en paralelo sobre la misma tarea y agregar resultados.

📚 **Doc oficial:** [Concurrent Orchestration (Python)](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/concurrent?pivots=programming-language-python)

## ¿Qué es Concurrent Orchestration?

Todos los agentes procesan el **mismo input al mismo tiempo** y un agregador combina sus salidas.

```
                ┌─→ [Researcher] ─┐
Usuario  ───────┼─→ [Marketer]   ─┼──→  Aggregator  ──→  AgentResponse final
                └─→ [Legal]      ─┘
```

Ideal para:
- Brainstorming desde múltiples perspectivas
- Ensemble reasoning / voting
- Análisis multi-disciplinario (legal + técnico + marketing)

## API clave

```python
from agent_framework.orchestrations import ConcurrentBuilder

workflow = ConcurrentBuilder(participants=[a, b, c]).build()
events = await workflow.run("...")
outputs = events.get_outputs()     # [AgentResponse]  — un solo objeto con N mensajes
```

> El agregador por defecto produce **un `AgentResponse`** con un mensaje por participante.
> Puedes pasar un agregador custom con `.with_aggregator(callback)`.


## ⚙️ Setup

In [1]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados.")


C:\Users\jreyesleger\AppData\Roaming\Python\Python312\site-packages\agent_framework\_skills.py:116: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
C:\Users\jreyesleger\AppData\Roaming\Python\Python312\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


✅ Helpers cargados.


## 1️⃣ Fan-out básico con 3 expertos

Tres agentes con expertise distinta analizan el mismo prompt **en paralelo**.
El framework hace fan-out, espera a todos y agrega.


In [ ]:
from agent_framework import AgentResponse
from agent_framework.orchestrations import ConcurrentBuilder

client = create_chat_client()

researcher = Agent(
    client=client,
    name="researcher",
    instructions=(
        "Eres un investigador experto de mercado y productos. Dado un prompt, proporciona información concisa, "
        "información factual, oportunidades y riesgos. Responde en español."
    ),
)

marketer = Agent(
    client=client,
    name="marketer",
    instructions=(
        "Eres un estratega de marketing creativo. Elabora propuestas de valor convincentes y "
        "mensajes dirigidos alineados al prompt. Responde en español."
    ),
)

legal = Agent(
    client=client,
    name="legal",
    instructions=(
        "Eres un revisor cauteloso de aspectos legales y cumplimiento. Destaca restricciones, descargos de responsabilidad, "
        "y preocupaciones de políticas basadas en el prompt. Responde en español."
    ),
)

workflow = ConcurrentBuilder(participants=[researcher, marketer, legal]).build()

events = await workflow.run(
    "Estamos lanzando una nueva bicicleta eléctrica económica para viajeros urbanos."
)
outputs = events.get_outputs()

print("===== Final Aggregated Results =====\n")
if outputs:
    # outputs[0] es una lista de Message (un mensaje por participante)
    messages = outputs[0] if isinstance(outputs[0], list) else [outputs[0]]
    for msg in messages:
        name = msg.author_name or "assistant"
        print(f"{'-' * 60}\n[{name}]:\n{msg.text}\n")


===== Final Aggregated Results =====

------------------------------------------------------------
[assistant]:
We are launching a new budget-friendly electric bike for urban commuters.

------------------------------------------------------------
[researcher]:
### Insights:
1. **Market Growth**: The global e-bike market is projected to reach $80 billion by 2027, driven by urbanization, rising fuel prices, and eco-conscious consumer behavior.
2. **Urban Commuters**: Urban dwellers seek cost-efficient, sustainable, and time-saving modes of transportation. E-bikes are increasingly seen as a viable alternative to cars, public transport, and scooters.
3. **Key Features for Target Market**: Budget-conscious urban commuters prioritize affordability, low maintenance costs, lightweight design, and decent battery range (30-50 miles).

### Opportunities:
1. **Affordability Appeal**: By emphasizing low pricing without compromising on quality, you can tap into a growing demand for accessible e-bik

## 2️⃣ Agregador custom — síntesis con LLM

Por defecto recibes los outputs concatenados. Si quieres que **otro LLM resuma** o consolide
todos los outputs en un solo texto, pasa un callback async a `.with_aggregator(...)`.


In [ ]:
from agent_framework import AgentExecutorResponse

summarizer_agent = Agent(
    client=client,
    name="summarizer",
    instructions=(
        "Consolidas los resultados de múltiples expertos de dominio en un resumen cohesivo y conciso "
        "con conclusiones claras. Mantenlo en menos de 200 palabras. Responde en español."
    ),
)


async def summarize_results(results: list[AgentExecutorResponse]) -> str:
    """Toma los outputs de todos los expertos y los pasa al summarizer."""
    sections: list[str] = []
    for r in results:
        messages = getattr(r.agent_response, "messages", [])
        final_text = messages[-1].text if messages and hasattr(messages[-1], "text") else "(no content)"
        sections.append(f"{r.executor_id}:\n{final_text}")

    prompt = "\n\n".join(sections)
    response = await summarizer_agent.run(prompt)
    return response.messages[-1].text if response.messages else ""


workflow = (
    ConcurrentBuilder(participants=[researcher, marketer, legal])
    .with_aggregator(summarize_results)
    .build()
)

output_text = None
async for event in workflow.run(
    "Estrategia de lanzamiento para una nueva bicicleta eléctrica",
    stream=True,
):
    if event.type == "output":
        output_text = event.data

print("===== Final Consolidated Output =====\n")
print(output_text)


## 3️⃣ Streaming + intermediate outputs

Por defecto sólo el agregador emite `"output"`. Si quieres ver el output de cada participante
**a medida que llega**, usa `intermediate_output_from=[...]` y observa los eventos `"intermediate"`.


In [ ]:
from agent_framework import AgentResponseUpdate

workflow = ConcurrentBuilder(
    participants=[researcher, marketer, legal],
    intermediate_outputs=True,
).build()

last_author = None
async for event in workflow.run(
    "Analiza nuestra nueva estrategia de lanzamiento de producto.",
    stream=True,
):
    if event.type == "intermediate" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"[{author}]: {update.text}", end="", flush=True)
            last_author = author
        else:
            print(update.text, end="", flush=True)
print()


## 🎯 Resumen

| Capacidad | API |
|-----------|-----|
| Fan-out paralelo | `ConcurrentBuilder(participants=[a, b, c]).build()` |
| Agregador custom | `.with_aggregator(async_callback)` |
| Outputs intermedios | `intermediate_output_from=[...]` |
| Streaming | `async for event in workflow.run(prompt, stream=True):` |

**Siguiente módulo →** [Módulo 11 — Orchestration Handoff](./11_orchestration_handoff.ipynb)
